# Train Network Basics

In this notebook, you will learn the fundamentals of the train network simulation:

1. Load train configuration from `config.yaml`
2. Create train and station objects
3. Manually simulate a train moving through one route cycle
4. Publish train position updates to MQTT

This forms the foundation for understanding how trains operate in our simulation.

## Step 1: Load Configuration

First, we load the train network configuration which includes:
- Train specifications (capacity, base occupancy)
- Route definition (stations in order)
- MQTT settings

In [ ]:
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher

# Load configuration
config = load_config()

# Display train network config
print("Train Configuration:")
print(f"  Capacity: {config.train_network.train.capacity} passengers")
print(f"  Base Occupancy: {config.train_network.train.base_occupancy_percent}%")
print(f"  Departure Interval: {config.train_network.train.departure_interval_minutes} minutes")
print(f"\nRoute ({len(config.train_network.route)} stations):")
for i, station in enumerate(config.train_network.route):
    print(f"  {i+1}. {station.name} ({station.type})")

## Step 2: Connect to MQTT Broker

We need MQTT to publish train status updates that can be monitored by other parts of the system.

In [ ]:
# Connect to MQTT broker
connector = MqttConnector(config.mqtt, client_id_suffix="train-basics")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
    publisher = MqttPublisher(connector)
else:
    print("✗ Failed to connect to MQTT broker")

## Step 3: Create Station Objects

Convert the configuration into `Station` objects that our agents can use.

In [ ]:
from simulated_city.agents import Station

# Convert config stations to Station objects
route = []
for station_config in config.train_network.route:
    station = Station(
        name=station_config.name,
        station_type=station_config.type,
        location_lat=station_config.location.lat,
        location_lon=station_config.location.lon,
        exit_percentage=station_config.exit_percentage,
    )
    route.append(station)

print(f"Created {len(route)} station objects")
for station in route:
    print(f"  {station.name}: ({station.location_lat:.4f}, {station.location_lon:.4f})")

## Step 4: Create a Train

Now we create a single train that will travel through our route.

**Key concepts:**
- **Capacity**: Maximum number of passengers the train can hold
- **Base occupancy**: Non-event passengers always on the train (background load)
- **Current station index**: Which station the train is at in the route

In [ ]:
from simulated_city.agents import Train, TrainStatus

# Calculate base occupancy count
capacity = config.train_network.train.capacity
base_occupancy_percent = config.train_network.train.base_occupancy_percent
base_occupancy_count = int(capacity * base_occupancy_percent / 100)

# Create train
train = Train(
    id="train-001",
    capacity=capacity,
    current_station_index=0,
    current_station_name=route[0].name,
    base_occupancy_count=base_occupancy_count,
    status=TrainStatus.AT_STATION,
)

print(f"Train {train.id} created:")
print(f"  Capacity: {train.capacity}")
print(f"  Base occupancy: {train.base_occupancy_count} passengers")
print(f"  Available capacity: {train.available_capacity}")
print(f"  Current location: {train.current_station_name}")
print(f"  Status: {train.status.value}")

## Step 5: Create a Train Agent

The `TrainAgent` manages a train's operations including movement and passenger handling.

In [ ]:
from simulated_city.agents import TrainAgent

# Create train agent
train_agent = TrainAgent(
    train=train,
    route=route,
    mqtt_publisher=publisher,
    mqtt_base_topic=config.train_network.mqtt_base_topic,
)

print(f"TrainAgent created for {train.id}")
print(f"  Route has {len(route)} stations")
print(f"  Publishing to topic: {config.train_network.mqtt_base_topic}/train/{train.id}/status")

## Step 6: Manually Simulate One Route Cycle

Let's manually move the train through each station and publish its status.

**For each station:**
1. Update train position
2. Set appropriate status (AT_STATION, IN_TRANSIT)
3. Publish status to MQTT

In [ ]:
import time

print("Starting route cycle...\n")

for station_index, station in enumerate(route):
    # Update train position
    train.current_station_index = station_index
    train.current_station_name = station.name
    train.status = TrainStatus.AT_STATION
    
    # Publish status
    print(f"Station {station_index + 1}/{len(route)}: {station.name}")
    print(f"  Status: {train.status.value}")
    print(f"  Passengers: {train.total_passengers}/{train.capacity}")
    train_agent.publish_status()
    print(f"  ✓ Status published to MQTT")
    
    # Simulate dwell time at station
    time.sleep(1)
    
    # Move to next station (if not last)
    if station_index < len(route) - 1:
        train.status = TrainStatus.IN_TRANSIT
        print(f"  → Moving to next station...\n")
        train_agent.publish_status()
        time.sleep(1)  # Simulate travel time

print("\n✓ Route cycle complete!")

## Step 7: Inspect Published Messages

The train has published status messages to MQTT. Open another notebook or MQTT client to subscribe to:

```
train_network/train/train-001/status
```

You should see JSON messages with train position, capacity, and status.

## Cleanup

Disconnect from MQTT when done.

In [ ]:
# Disconnect from MQTT
if connector.client and connector.client.is_connected():
    connector.disconnect()
    print("✓ Disconnected from MQTT broker")

## Exercise: Experiment with Train Properties

Try modifying the train and re-running the simulation:

1. **Change base occupancy**: What happens if the train starts with more passengers?
2. **Add event passengers**: Manually add some `Passenger` objects to `train.passengers_onboard`
3. **Monitor capacity**: Print `train.capacity_percentage` at each station

Example:
```python
from simulated_city.agents import Passenger
from datetime import datetime

# Add 100 event passengers
for i in range(100):
    passenger = Passenger(
        id=f"p-{i}",
        entry_station=route[0].name,
        exit_station=route[-1].name,
        arrival_time=datetime.now(),
    )
    train.passengers_onboard.append(passenger)

print(f"Train now has {train.total_passengers} passengers ({train.capacity_percentage:.1f}% capacity)")
```

## Next Steps

Continue to **05_train_passenger_source.ipynb** to learn how passengers are generated and queued at stations.